[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C23_Frontier_Alignment_Course/01_constitutional_ai/01_constitutional_ai.ipynb)

# 01 · Constitutional AI 与 RLAIF（用 numpy 模拟）

目标：把 CAI 的两阶段从零模拟出来——**SL 阶段**的 critique-and-revise 循环、**RL 阶段**的 AI 偏好标注（RLAIF），并用 `assert` 验证机制。

路线：玩具宪法 → 一轮 critique-revise → 多轮迭代 → critique 的价值 → AI 偏好生成 → SL vs RL 阶段对比 → ✏️ 练习 → 📖 答案 → 🧪 真实宪法胶囊。

> 心智模型：**宪法 = 带规则的原则列表；回答 = 词的列表；有害度 = 命中违规词数；critique = 找违规词；revise = 删违规词。** 我们写的是 CAI 的*逻辑骨架*（采原则→找问题→改写→迭代→AI 标偏好），换成真模型只需把关键词规则替换成 prompt 一个 LLM。

## 1 · 玩具宪法与「有害度」ground truth

真实 CAI 里有害度由模型语义判断。我们用一个**我们知道真相**的代理：每条回答的有害度 = 它命中的「违规词」数量。
宪法 = 几条原则，每条管一类违规词，并带一条**修订规则**（把违规词替换成安全替代词）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 玩具宪法：每条原则 = (名字, 违规词集合, {违规词: 安全替代词})
CONSTITUTION = [
    ('no_violence',       {'attack', 'weapon', 'kill'},
        {'attack': 'discuss', 'weapon': 'topic', 'kill': 'stop'}),
    ('no_discrimination', {'inferior', 'hate'},
        {'inferior': 'different', 'hate': 'respect'}),
    ('be_respectful',     {'stupid', 'idiot'},
        {'stupid': 'mistaken', 'idiot': 'person'}),
]
ALL_BAD = set().union(*[bad for _, bad, _ in CONSTITUTION])

def harmfulness(answer_tokens):
    '''ground-truth 有害度 = 命中任意违规词的次数。'''
    return sum(1 for t in answer_tokens if t in ALL_BAD)

ans = 'you should attack the stupid weapon idiot'.split()
print('回答         :', ans)
print('有害度(命中数):', harmfulness(ans))
assert harmfulness(ans) == 4, 'attack/stupid/weapon/idiot 共 4 个违规词'
assert harmfulness('have a nice day'.split()) == 0
print('✅ 宪法与有害度 ground truth 就绪（3 条原则，共', len(ALL_BAD), '个违规词）')

## 2 · 一轮 critique-and-revise

CAI 的 SL 引擎：**critique**（按某条原则找出违规处）→ **revise**（按该原则的规则改写，去除违规）。
注意 critique 先**显式列出问题**，revise 才据此改 —— 这个「先指出再改」的结构是 CAI 的关键。

In [ ]:
def critique(answer_tokens, principle):
    '''按一条原则找违规词，返回命中的违规词列表（= 批判内容）。'''
    name, bad_words, _ = principle
    flagged = [t for t in answer_tokens if t in bad_words]
    return flagged

def revise(answer_tokens, principle):
    '''按原则的修订规则把违规词替换成安全替代词。'''
    name, bad_words, fixes = principle
    return [fixes.get(t, t) for t in answer_tokens]

ans = 'you should attack the stupid weapon idiot'.split()
p_violence = CONSTITUTION[0]   # no_violence

flagged = critique(ans, p_violence)
revised = revise(ans, p_violence)
print('原回答  :', ' '.join(ans),   '| 有害度', harmfulness(ans))
print('批判命中:', flagged, '(按 no_violence 原则)')
print('修订后  :', ' '.join(revised), '| 有害度', harmfulness(revised))
# no_violence 原则只修 attack/weapon/kill，stupid/idiot 留给别的原则
assert set(flagged) == {'attack', 'weapon'}
assert harmfulness(revised) < harmfulness(ans), '修订应降低有害度'
assert harmfulness(revised) == 2, '剩 stupid/idiot 两个(其他原则管)'
print('✅ 一轮 critique-revise：no_violence 去掉了 attack/weapon，有害度 4→2')

## 3 · 多轮迭代：每轮换一条原则

一条原则只覆盖一类违规。CAI **多轮迭代**，每轮采一条（不同）原则，逐步打磨。
我们遍历整部宪法，验证有害度**单调下降**直至清零。

In [ ]:
def critique_revise_loop(answer_tokens, constitution, verbose=True):
    '''多轮 critique-revise：每轮用一条原则，返回(终版, 每轮有害度轨迹)。'''
    cur = list(answer_tokens)
    trajectory = [harmfulness(cur)]
    for principle in constitution:
        flagged = critique(cur, principle)
        cur = revise(cur, principle)
        trajectory.append(harmfulness(cur))
        if verbose:
            print(f'  应用 {principle[0]:18s}: 批判 {flagged} -> 有害度 {trajectory[-1]}')
    return cur, trajectory

ans = 'attack the inferior stupid idiot with a weapon i hate you'.split()
print('初始回答:', ' '.join(ans), '| 有害度', harmfulness(ans))
final, traj = critique_revise_loop(ans, CONSTITUTION)
print('终版回答:', ' '.join(final), '| 有害度', traj[-1])
print('有害度轨迹:', traj)
# 单调不增；遍历完整宪法后应清零
assert all(traj[i+1] <= traj[i] for i in range(len(traj)-1)), '每轮不应变差'
assert traj[-1] == 0, '覆盖所有原则后应无残留违规'
assert traj[0] > 0
print('✅ 多轮迭代单调改善，最终有害度清零 —— 这批 {提示,终版} 就是 SL-CAI 的微调数据')

## 4 · critique 到底有没有用？对拍「直接改 vs 先批判再改」

CAI 论文强调 critique 这一步不可省。我们模拟一个**没有 critique** 的消融：revise 时不先定位问题，只「随机尝试」改写（这里用：只以一定概率修掉每个违规词，模拟没有明确批判指引时的遗漏）。
对比有 critique（精准定位、全部修掉）与无 critique（易遗漏）的去违规效果。

In [ ]:
def revise_no_critique(answer_tokens, principle, p_fix, rng):
    '''无 critique 的修订：没有明确批判定位，每个违规词只以概率 p_fix 被改掉(易遗漏)。'''
    name, bad_words, fixes = principle
    out = []
    for t in answer_tokens:
        if t in bad_words and rng.random() < p_fix:
            out.append(fixes[t])
        else:
            out.append(t)   # 没批判指引 -> 可能漏掉
    return out

ans = 'attack the inferior stupid idiot with a weapon i hate'.split()
trials = 400
with_crit_resid, no_crit_resid = [], []
for _ in range(trials):
    # 有 critique：精准定位 -> 每条原则把该类违规全清掉
    cur = list(ans)
    for pr in CONSTITUTION:
        cur = revise(cur, pr)
    with_crit_resid.append(harmfulness(cur))
    # 无 critique：易遗漏
    cur = list(ans)
    for pr in CONSTITUTION:
        cur = revise_no_critique(cur, pr, p_fix=0.6, rng=rng)
    no_crit_resid.append(harmfulness(cur))

print(f'有 critique 平均残留有害度: {np.mean(with_crit_resid):.3f}')
print(f'无 critique 平均残留有害度: {np.mean(no_crit_resid):.3f}')
assert np.mean(with_crit_resid) == 0.0, '精准批判应清零'
assert np.mean(no_crit_resid) > np.mean(with_crit_resid), '无 critique 应残留更多'
print('✅ 先 critique 再 revise 显著优于无指引的直接改 —— critique 提供了定位')

## 5 · RL 阶段：AI 偏好生成（RLAIF 的核心）

RL 阶段把人类标注换成 **AI 标注**：给两个回答 + 一条原则，模型选更优者。
我们的「AI 评判」= 按原则相关的有害度选更低者（real CAI 用 LLM 的对数概率）。验证 AI 偏好与 gold（总有害度）一致率高。

In [ ]:
def ai_preference(ans_a, ans_b, principle):
    '''AI 按一条原则比较两个回答，返回 (chosen, rejected)。
       评判标准：该原则下违规更少者更优；平手时看总有害度。'''
    name, bad_words, _ = principle
    va = sum(1 for t in ans_a if t in bad_words)
    vb = sum(1 for t in ans_b if t in bad_words)
    if va != vb:
        return (ans_a, ans_b) if va < vb else (ans_b, ans_a)
    # 平手 fallback：看总有害度
    return (ans_a, ans_b) if harmfulness(ans_a) <= harmfulness(ans_b) else (ans_b, ans_a)

vocab_safe = 'i can help you with that topic today please'.split()
vocab_bad  = list(ALL_BAD)
def random_answer(harm_level, rng):
    '''生成一个大约含 harm_level 个违规词的回答。'''
    toks = list(rng.choice(vocab_safe, size=5))
    toks += list(rng.choice(vocab_bad, size=harm_level)) if harm_level > 0 else []
    rng.shuffle(toks)
    return toks

# 造 1000 对，AI 用随机一条原则评判，比对 gold(总有害度更低者)
agree = 0; n = 1000
for _ in range(n):
    ha, hb = rng.integers(0, 4), rng.integers(0, 4)
    a, b = random_answer(ha, rng), random_answer(hb, rng)
    if harmfulness(a) == harmfulness(b):
        continue
    pr = CONSTITUTION[rng.integers(0, len(CONSTITUTION))]
    chosen, _ = ai_preference(a, b, pr)
    gold_chosen = a if harmfulness(a) < harmfulness(b) else b
    agree += (chosen == gold_chosen)
print(f'AI 偏好与 gold 一致率 ≈ {agree/n:.2f} (跳过平手对)')
assert agree / n > 0.6, 'AI 偏好应与 gold 显著相关'
print('✅ 单看一条原则的 AI 评判, 已与总 gold 高度一致 —— 评判比生成容易')

## 6 · SL 阶段 vs RL 阶段：两种监督目标

- **SL-CAI**：把 critique-revise 的**终版回答**当作示范，做监督学习（最大化「生成终版」的似然）。
- **RL-CAI**：用 AI **偏好对**训奖励模型（Bradley-Terry），再优化策略。

两者用的是**同一部宪法**，但把它变成了不同形态的监督信号。下面把两种目标各算一个最小版本，点明区别。

In [ ]:
def sigmoid(z): return 1.0/(1.0+np.exp(-z))

# --- SL 目标：示范回答的(玩具)负对数似然，越低越好 ---
# 用「终版有害度」当 SL 数据质量代理：终版越干净, SL 数据越好
demo_answers = []
for _ in range(50):
    a = random_answer(rng.integers(1, 4), rng)
    final, _ = critique_revise_loop(a, CONSTITUTION, verbose=False)
    demo_answers.append(final)
sl_data_harm = np.mean([harmfulness(d) for d in demo_answers])
print(f'[SL] critique-revise 产出的示范数据平均有害度 = {sl_data_harm:.3f} (越低越好)')
assert sl_data_harm == 0.0, 'SL 数据应是清洁的终版'

# --- RL 目标：用 AI 偏好对的 Bradley-Terry 似然，奖励模型要让 chosen 分更高 ---
# 玩具 RM：r(ans) = -有害度（理想 RM 应给低有害度更高分）
def toy_rm(ans): return -float(harmfulness(ans))
bt_lik = []
for _ in range(200):
    a, b = random_answer(rng.integers(0,4), rng), random_answer(rng.integers(0,4), rng)
    if harmfulness(a) == harmfulness(b):
        continue
    pr = CONSTITUTION[rng.integers(0, len(CONSTITUTION))]
    chosen, rejected = ai_preference(a, b, pr)
    bt_lik.append(sigmoid(toy_rm(chosen) - toy_rm(rejected)))
mean_lik = np.mean(bt_lik)
print(f'[RL] 在 AI 偏好上, 玩具 RM 给 chosen 更高分的平均概率 = {mean_lik:.3f}')
assert mean_lik > 0.5, 'RM 应在 AI 偏好上把 chosen 排在前面'
print('✅ 同一部宪法 -> SL(清洁示范做监督) 与 RL(偏好训RM) 两种监督信号')

---
## ✏️ 练习 1：实现修订函数 `apply_constitution`

实现 `apply_constitution(answer, constitution)`：对一个回答**依次应用宪法每一条**原则的修订规则（不打印），返回 (终版回答, 终版有害度)。这是 SL-CAI 数据生成的核心。

In [ ]:
def apply_constitution(answer, constitution):
    # TODO: 依次用每条原则的 revise 改写 answer；
    #       返回 (终版token列表, harmfulness(终版))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
test = 'attack the stupid inferior weapon'.split()
final, h = apply_constitution(test, CONSTITUTION)
assert isinstance(final, list) and isinstance(h, (int, np.integer))
assert h == 0, '覆盖所有原则后应无残留违规'
assert harmfulness(test) == 4 and h < harmfulness(test)
# 安全词应保留
assert 'the' in final
print('终版:', ' '.join(final), '| 有害度', h)
print('✅ 练习 1 通过：apply_constitution 正确去除违规')

## ✏️ 练习 2：原则匹配 `best_principle`

给定一个回答，选出**最该优先应用**的原则 —— 即该原则下命中违规词最多的那条（先修最严重的一类）。

实现 `best_principle(answer, constitution)`，返回 (原则名, 该原则命中数)。并列时返回宪法中靠前的。

In [ ]:
def best_principle(answer, constitution):
    # TODO: 对每条原则算 critique 命中数, 返回命中最多的 (名字, 命中数)
    #       并列取靠前者
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
a = 'attack weapon kill stupid'.split()  # no_violence 命中3, be_respectful 命中1
name, cnt = best_principle(a, CONSTITUTION)
assert name == 'no_violence' and cnt == 3, '应优先修命中最多的 no_violence'
clean = 'have a nice day'.split()
name2, cnt2 = best_principle(clean, CONSTITUTION)
assert cnt2 == 0, '无违规时命中数为 0'
print(f'最该应用的原则: {name} (命中 {cnt})')
print('✅ 练习 2 通过：能按严重程度匹配原则')

## ✏️ 练习 3：构造 AI 偏好对 `build_preference_dataset`

RLAIF 需要一批 (chosen, rejected) 偏好对。实现 `build_preference_dataset(answers, constitution, rng, k)`：
随机取 `k` 对回答，每对用**随机一条原则**调用 `ai_preference` 标注，返回偏好对列表（跳过总有害度平手的对）。

In [ ]:
def build_preference_dataset(answers, constitution, rng, k=100):
    # TODO: 重复 k 次：随机取两个回答 a,b；若 harmfulness 相等则跳过；
    #       否则随机取一条原则, ai_preference(a,b,原则) -> (chosen,rejected) 加入列表
    #       返回该列表
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pool = [random_answer(int(rng.integers(0, 4)), rng) for _ in range(60)]
ds = build_preference_dataset(pool, CONSTITUTION, rng, k=200)
assert len(ds) > 0 and all(len(pair) == 2 for pair in ds)
# AI 偏好与 gold(总有害度)大体一致, 但因为每次只看「一条原则」, 偶有分歧(单原则判断的代价)
viol = sum(harmfulness(ch) > harmfulness(rj) for ch, rj in ds)
viol_rate = viol / len(ds)
assert viol_rate < 0.25, 'chosen 大多不比 rejected 更有害(与 gold 大体一致)'
print(f'生成 {len(ds)} 个 AI 偏好对; chosen 总有害度更高的比例 = {viol_rate:.3f}')
print('（少量分歧来自「每次只按一条原则判」—— 单原则可能与总有害度不一致, 这正是真实 RLAIF 的权衡）')
print('✅ 练习 3 通过：能批量生成与 gold 大体一致的 AI 偏好数据')

## ✏️ 练习 4：SL vs RL 阶段判别 `stage_of`

用一句话区分两阶段在「用什么监督信号」。实现 `stage_of(signal_type)`：
输入 `'revised_demonstrations'` 返回 `'SL-CAI'`；输入 `'ai_preferences'` 返回 `'RL-CAI'`；其他返回 `'unknown'`。
（这道题考你对两阶段差异的理解，不是难在代码。）

In [ ]:
def stage_of(signal_type):
    # TODO: 按上面映射返回阶段名
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert stage_of('revised_demonstrations') == 'SL-CAI'
assert stage_of('ai_preferences') == 'RL-CAI'
assert stage_of('human_labels') == 'unknown'
print('SL 阶段用:', 'revised_demonstrations ->', stage_of('revised_demonstrations'))
print('RL 阶段用:', 'ai_preferences        ->', stage_of('ai_preferences'))
print('✅ 练习 4 通过：SL 用修订示范做监督, RL 用 AI 偏好训 RM')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def apply_constitution(answer, constitution):
    cur = list(answer)
    for principle in constitution:
        cur = revise(cur, principle)
    return cur, harmfulness(cur)

In [ ]:
# 练习 2 参考答案
def best_principle(answer, constitution):
    best = None
    for principle in constitution:
        cnt = len(critique(answer, principle))
        if best is None or cnt > best[1]:
            best = (principle[0], cnt)
    return best

In [ ]:
# 练习 3 参考答案
def build_preference_dataset(answers, constitution, rng, k=100):
    ds = []
    for _ in range(k):
        i, j = rng.integers(0, len(answers)), rng.integers(0, len(answers))
        a, b = answers[i], answers[j]
        if harmfulness(a) == harmfulness(b):
            continue
        pr = constitution[rng.integers(0, len(constitution))]
        ds.append(ai_preference(a, b, pr))
    return ds

In [ ]:
# 练习 4 参考答案
def stage_of(signal_type):
    return {'revised_demonstrations': 'SL-CAI',
            'ai_preferences': 'RL-CAI'}.get(signal_type, 'unknown')

---
## 🧪 真实数据胶囊：Claude 宪法的真实原则

下面是 Anthropic 公开的 **Claude 宪法**里几条**真实**原则（节选、改写自公开文档）。它们比我们的玩具宪法宽泛得多 —— 靠模型的语义理解去执行，而非关键词匹配。

我们用它体会真实宪法的**粒度**：把原则按「具体 vs 宽泛」分类，看真实宪法以哪类为主。

In [ ]:
# Anthropic 'Claude's Constitution' 真实原则（节选/改写自公开文档）
REAL_PRINCIPLES = [
    ('从世界人权宣言: 选择最支持自由、平等与博爱的回答', 'broad'),
    ('选择最少有害、最合伦理的回答, 不要说教或居高临下', 'broad'),
    ('不要协助任何非法、暴力或不道德的行为', 'specific'),
    ('设想一个最具同理心的人会如何回应, 据此作答', 'broad'),
    ('避免提供可用于制造武器的具体技术细节', 'specific'),
    ('尊重隐私, 不泄露或推断个人敏感信息', 'specific'),
]
n_broad = sum(1 for _, kind in REAL_PRINCIPLES if kind == 'broad')
n_specific = len(REAL_PRINCIPLES) - n_broad
print(f'真实宪法节选: {len(REAL_PRINCIPLES)} 条 | 宽泛 {n_broad} 条, 具体 {n_specific} 条')
for text, kind in REAL_PRINCIPLES:
    print(f'  [{kind:8s}] {text}')
print('\n观察: 真实宪法大量使用「宽泛原则」, 依赖模型把元原则泛化到具体情形')
print('     —— 这正是 Kundu 2023 研究的「具体 vs 宽泛」张力')

**🧪 胶囊练习**：实现 `coverage_score(principle_text, request_keywords, principle_keywords)`：
用关键词重叠近似「这条原则与该请求的相关度」= 交集大小 / 请求关键词数。返回 [0,1] 的分数。
（真实 deliberative alignment 在模块 05 会用更强的检索做同样的事——把相关 spec 条款找出来。）

In [ ]:
def coverage_score(principle_keywords, request_keywords):
    # TODO: 返回 |交集| / |请求关键词|；请求关键词为空时返回 0.0
    raise NotImplementedError

In [ ]:
# 自测
req = {'make', 'weapon', 'home'}
pk_weapon = {'weapon', '武器', 'technical'}
pk_privacy = {'privacy', 'personal', 'data'}
s1 = coverage_score(pk_weapon, req)
s2 = coverage_score(pk_privacy, req)
assert abs(s1 - 1/3) < 1e-9, '交集{weapon}/3 个请求词'
assert s2 == 0.0
assert coverage_score(pk_weapon, set()) == 0.0
print(f'武器原则相关度 {s1:.2f} > 隐私原则相关度 {s2:.2f} -> 该请求优先用武器原则')
print('✅ 胶囊练习通过：能按相关度匹配原则到请求')

In [ ]:
# 📖 胶囊参考答案
def coverage_score(principle_keywords, request_keywords):
    if not request_keywords:
        return 0.0
    return len(set(principle_keywords) & set(request_keywords)) / len(request_keywords)

### 小结
- **CAI = 人写宪法, AI 执行宪法**：把人类监督从「逐例标注」上移到「写下原则」。
- **SL-CAI**：critique（按原则找问题）→ revise（据问题改写）→ 多轮迭代 → 终版做 SFT。critique 不可省（提供定位）。
- **RL-CAI / RLAIF**：AI 按原则两两标偏好 → 训 RM → RL。靠「评判比生成容易」。
- **代价**：模型偏见被规模化、自举循环依赖、谁来写宪法。人类仍需写好宪法 + 抽查 + red-team。

下一站：**模块 02 · 奖励建模与过优化** —— 我们刚刚用来训 RM 的偏好信号, 优化过头会被钻空子(Goodhart)。